# FinBERT(sentiment)

In [ ]:
import pandas as pd

df = pd.read_csv('ESG_Dataset_13000/all_articles_title_dedup.csv')

df['publication_date'] = pd.to_datetime(df['publication_date'])

df.head()

,publication_date,document_title,author_name,source_name,text,source_file
0,2016-03-31,Sustainability - a hot item on board agendasSu...,NaN,NaN,"TODAY, over 3.9 billion or 54 per cent of the ...",EsgNews109.csv
1,2016-06-20,Deutsche Bank aims to raise EUR1bn through soc...,NaN,NaN,Lender anticipates surge in demand from Europe...,EsgNews124.csv
2,2016-08-01,Thailand: Thai bourse scores top liquidity and...,NaN,NaN,The Stock Exchange of Thailand (SET) has revea...,EsgNews92.csv
3,2016-08-07,Fund managers searching for sustainable betsFu...,NaN,NaN,"At SET Social Impact, an event hosted by the S...",EsgNews107.csv
4,2016-10-03,Investing goes fossil free; Divestment does no...,NaN,NaN,Put it down to the rise of the divestment move...,EsgNews119.csv


In [3]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

model_name = "yiyanghkust/finbert-tone"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

model.eval()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1162.27it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-tone
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30873, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [4]:
def split_text(text, max_len=510):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []

    for i in range(0, len(tokens), max_len):
        chunk_tokens = tokens[i:i+max_len]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)

    return chunks

In [5]:
import numpy as np

def get_chunk_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1).numpy()[0]

    neu, pos, neg = probs

    return pos, neg, neu

In [6]:
def get_article_sentiment(text):
    chunks = split_text(text)

    pos_list = []
    neg_list=[]
    neu_list=[]

    for chunk in chunks:
        pos, neg, neu = get_chunk_sentiment(chunk)
        pos_list.append(pos)
        neg_list.append(neg)
        neu_list.append(neu)

# calculate mean prob of each chunk
    pos = np.mean(pos_list)                 
    neg = np.mean(neg_list)         
    neu = np.mean(neu_list)              

    return pos,neg,neu

In [12]:
from tqdm import tqdm

pos = []
neg = []
neu = []

for text in tqdm(df['text']):
    s, i, u = get_article_sentiment(str(text))

    pos.append(s)
    neg.append(i)
    neu.append(u)

df['pos'] = pos
df['neg'] = neg
df['neu'] = neu
df['sentiment_article'] = df['pos'] - df['neg']
df['month'] = df['publication_date'].dt.to_period('M')

print(df[['publication_date', 'pos', 'neg', 'neu', 'sentiment_article', 'month']].to_string(index=False))


100%|██████████| 2194/2194 [16:13<00:00,  2.25it/s]


,publication_date,pos,neg,neu,sentiment_article,month
0,2016-03-31,0.466727,0.046095,0.487178,0.420633,2016-03
1,2016-06-20,0.372524,0.000155,0.627321,0.372369,2016-06
2,2016-08-01,0.400394,0.000044,0.599562,0.400350,2016-08
3,2016-08-07,0.000159,0.000023,0.999818,0.000136,2016-08
4,2016-10-03,0.001011,0.000983,0.998006,0.000028,2016-10


In [ ]:
# Monthly aggregation based on article-level probabilities and sentiment
monthly = df.groupby('month').agg(
    FinBERT_positive = ('pos', 'mean'),
    FinBERT_negative = ('neg', 'mean'),
    FinBERT_neutral  = ('neu', 'mean'),
    FinBERT_sentiment = ('sentiment_article', 'mean'),
    article_count    = ('sentiment_article', 'count'),
).reset_index()

print(f"Monthly rows: {len(monthly)}")
print(monthly.to_string(index=False))

Monthly rows: 112
  month  FinBERT_positive  FinBERT_negative  FinBERT_neutral  FinBERT_sentiment  article_count
2016-03          0.466727          0.046095         0.487178           0.420633              1
2016-06          0.372524          0.000155         0.627321           0.372369              1
2016-08          0.200276          0.000034         0.799690           0.200243              2
2016-10          0.001011          0.000983         0.998006           0.000028              1
2016-12          0.305327          0.002772         0.691901           0.302555              2
2017-02          0.000303          0.000261         0.999436           0.000042              1
2017-04          0.135235          0.000248         0.864517           0.134986              1
2017-06          0.108740          0.245125         0.646136          -0.136385              1
2017-07          0.166667          0.019515         0.813819           0.147152              3
2017-09          0.089904       

In [16]:
monthly_sentiment = df.groupby('month').agg(
    sentiment_mean = ('sentiment_article', 'mean'),
    sentiment_std  = ('sentiment_article', 'std'),
    sentiment_abs  = ('sentiment_article', lambda x: abs(x).mean()),
    n_articles     = ('sentiment_article', 'count')
).reset_index()
print(monthly_sentiment.to_string(index=False))
monthly_sentiment.to_csv(
    "/Users/helen/BASc/dissertation/coding/monthly_sentiment.csv",
    index=False
)

  month  sentiment_mean  sentiment_std  sentiment_abs  n_articles
2016-03        0.420633            NaN       0.420633           1
2016-06        0.372369            NaN       0.372369           1
2016-08        0.200243       0.282994       0.200243           2
2016-10        0.000028            NaN       0.000028           1
2016-12        0.302555       0.223911       0.302555           2
2017-02        0.000042            NaN       0.000042           1
2017-04        0.134986            NaN       0.134986           1
2017-06       -0.136385            NaN       0.136385           1
2017-07        0.147152       0.302721       0.183857           3
2017-09        0.063060       0.174702       0.113101           3
2017-10        0.361954       0.161992       0.361954           2
2017-11        0.040611       0.092330       0.041699           5
2017-12        0.053634       0.038409       0.053634           2
2018-01        0.293372            NaN       0.293372           1
2018-02   